In [1]:
import os

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"  # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

# paper stats

In [ ]:
import pandas as pd
import ast

for lang in ["de", "fr", "pt", "ko", "nl"]:
    for split in ["train", "dev", "test"]:
        df = pd.read_csv(f"../chat-task-2024-data/{split}/en-{lang}.csv")
        df["source_length"] = df["source"].apply(lambda x: len(x))
        if "tags" in df.columns:
            df["is_tagged"] = df["tags"].apply(lambda x: len(ast.literal_eval(x)) != 0)
            tag_counts = dict(df["is_tagged"].value_counts())
            percent_tagged = tag_counts[True] / len(df)
        else:
            percent_tagged = 0
        avg_turns = (
            df.groupby("doc_id").size().reset_index(name="count")["count"].mean()
        )
        print(
            split,
            lang,
            len(df),
            f"{df['source_length'].mean():.2f}",
            f"{avg_turns:.2f}",
            f"{percent_tagged*100:.2f}",
        )

In [ ]:
import json
import sys
from typing import Any, Callable

import numpy as np

import sys

sys.path.append("/mnt/data/sagrawal/wmt24-chat-translation/MuDA")
from muda.langs import create_tagger
from muda.metrics import compute_metrics

In [ ]:
import pandas as pd

split = "test"
lang = "en-de"
df = pd.read_csv(f"../tacl_results/bcontrast/{split}.{lang}.csv")

In [ ]:
en_df = df[df.source_language == "en"]
len(en_df)

In [ ]:
tagger = create_tagger(
    "de",
    align_model="bert-base-multilingual-cased",
    align_cachedir=None,
    cohesion_threshold=3,
)

preproc = tagger.preprocess(en_df["source"], en_df["reference"], en_df["doc_id"])
tagged_refs = []
for doc in zip(*preproc):
    tagged_doc = tagger.tag(
        *doc, phenomena=["lexical_cohesion", "formality", "verb_form", "pronouns"]
    )
    tagged_refs.append(tagged_doc)

In [ ]:
def flatten(xss):
    return [x for xs in xss for x in xs]


tagged_refs = flatten(tagged_refs)

In [ ]:
tagged = []

for x in tagged_refs:
    tagged_curr = False
    for y in x:
        if len(y.tags) != 0:
            tagged_curr = True
            break
    tagged.append(tagged_curr)

In [ ]:
df["source_length"] = df["source"].apply(lambda x: len(x))
avg_turns = df.groupby("doc_id").size().reset_index(name="count")["count"].mean()

percent_tagged = sum(tagged) / len(df)
print(
    split,
    lang,
    len(df),
    f"{df['source_length'].mean():.2f}",
    f"{avg_turns:.2f}",
    f"{percent_tagged*100:.2f}",
)

# Main results

In [2]:
def read_file(fname):
    output = []
    with open(fname) as f:
        for line in f:
            output.append(line.strip())
    return output

In [4]:
from comet import download_model, load_from_checkpoint

comet_metric = load_from_checkpoint(download_model("Unbabel/wmt22-comet-da"))

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.3.3. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../../../scratch-artemis/sweta/cache/models--Unbabel--wmt22-comet-da/snapshots/f49d328952c3470eff6bb6f545d62bfdb6e66304/checkpoints/model.ckpt`
Encoder model frozen.


In [3]:
def get_translations(
    df,
    lp,
    context_type,
    model_name,
    split="dev",
    engiene="vllm",
    data_name="wmt24_chat",
):
    source_lang = lp.split("-")[0]
    target_lang = lp.split("-")[1].replace("pt", "pt-br")

    try:
        xx_yy_generations = read_file(
            f"../generations/{context_type}/mt/{data_name}_{split}.{source_lang}-{target_lang}/{engiene}/{model_name}/generation.txt"
        )
        yy_xx_generations = read_file(
            f"../generations/{context_type}/mt/{data_name}_{split}.{target_lang}-{source_lang}/{engiene}/{model_name}/generation.txt"
        )
    except:
        xx_yy_generations = read_file(
            f"../generations/{context_type}/mt/{data_name}_{split.replace('test', 'test_blind')}.{source_lang}-{target_lang}/{engiene}/{model_name}/generation.txt"
        )
        yy_xx_generations = read_file(
            f"../generations/{context_type}/mt/{data_name}_{split.replace('test', 'test_blind')}.{target_lang}-{source_lang}/{engiene}/{model_name}/generation.txt"
        )

    assert len(xx_yy_generations) + len(yy_xx_generations) == len(df)
    translations = []
    j, k = 0, 0

    for i, row in df.iterrows():
        if row["lp"] == f"{source_lang}-{target_lang}":
            translations.append(xx_yy_generations[j])
            j += 1
        else:
            # nl-en and it-en are considered here
            translations.append(yy_xx_generations[k])
            k += 1
    return translations

In [6]:
import pandas as pd

split = "test"

lang = "de"
df = pd.read_csv(f"../tacl_review/bcontrast/{split}.en-{lang}.csv")
df["lp"] = df["source_language"] + "-" + df["target_language"]

fnames = {
    # "baseline": read_file(f"../chat-task-2024-data/{split}/en-{lang}.baseline.txt"),
    # "gpt-4o-context": get_translations(
    #     df,
    #     "en-" + lang,
    #     "full_context_no_template",
    #     "gpt-4o",
    #     split,
    #     engiene="open-ai",
    #     data_name="bcontrast",
    # ),
    # "gpt-4o-no-context": get_translations(
    #     df,
    #     "en-" + lang,
    #     "no_context_no_template",
    #     "gpt-4o",
    #     split,
    #     engiene="open-ai",
    #     data_name="bcontrast",
    # ),
    # "greedy-7b-w-context": get_translations(
    #     df,
    #     "en-" + lang,
    #     "full_context",
    #     "TowerInstruct-7B-v0.2",
    #     split,
    #     data_name="bcontrast",
    # ),
    # "greedy-7b-wo-context": get_translations(
    #     df,
    #     "en-" + lang,
    #     "no_context",
    #     "TowerInstruct-7B-v0.2",
    #     split,
    #     data_name="bcontrast",
    # ),
    # "greedy-7b-ft-w-context": get_translations(
    #     df,
    #     "en-" + lang,
    #     "full_context_empty_sys",
    #     "TowerInstruct-7B-w-chat-empty-sys",
    #     split,
    #     data_name="bcontrast",
    # ),
    # "greedy-7b-ft-wo-context": get_translations(
    #     df,
    #     "en-" + lang,
    #     "no_context_empty_sys",
    #     "TowerInstruct-7B-w-chat-empty-sys",
    #     split,
    #     data_name="bcontrast",
    # ),
    # "mbr-instruct": read_file(
    #     f"../mbr_outputs/mbr_outputs_{split}/no_context/TowerInstruct-7B-v0.2/bcontrast_{split}.en-{lang}/comet_eps.out.txt"
    # ),
    # "mbr-source-instruct": read_file(
    #     f"../mbr_outputs/mbr_outputs_{split}/no_context/TowerInstruct-7B-v0.2/bcontrast_{split}.en-{lang}/comet_eps_context_source.out.txt"
    # ),
    # "mbr-chat": read_file(
    #     f"../mbr_outputs/mbr_outputs_{split}/full_context_empty_sys/TowerInstruct-7B-w-chat-empty-sys/bcontrast_{split}.en-{lang}/comet_eps.out.txt"
    # ),
    # "mbr-source-chat": read_file(
    #     f"../mbr_outputs/mbr_outputs_{split}/full_context_empty_sys/TowerInstruct-7B-w-chat-empty-sys/bcontrast_{split}.en-{lang}/comet_eps_context_source.out.txt"
    # ),
    # "towerchat-ft-7b-w-context": get_translations(
    #     df,
    #     "en-" + lang,
    #     "full_context_empty_sys",
    #     "TowerChat-1-epoch-ctx-comet-mbr-distilled",
    #     split,
    #     data_name="bcontrast",
    # ),
    # "towerchat-ft-7b-wo-context": get_translations(
    #     df,
    #     "en-" + lang,
    #     "no_context_empty_sys",
    #     "TowerChat-1-epoch-ctx-comet-mbr-distilled",
    #     split,
    #     data_name="bcontrast",
    # ),
    # "towerchat-plus-7b-w-context": get_translations(
    #     df,
    #     "en-" + lang,
    #     "full_context_empty_sys",
    #     "TowerChat-retrain-ctx-comet-mbr-distilled",
    #     split,
    #     data_name="bcontrast",
    # ),
    # "towerchat-plus-7b-wo-context": get_translations(
    #     df,
    #     "en-" + lang,
    #     "no_context_empty_sys",
    #     "TowerChat-retrain-ctx-comet-mbr-distilled",
    #     split,
    #     data_name="bcontrast",
    # ),
    "greedy-7b-ft-w-context-english": get_translations(
        df,
        "en-" + lang,
        "english_context_empty_sys",
        "TowerInstruct-7B-w-chat-empty-sys",
        split,
        data_name="bcontrast",
    ),
}

# for mname, file_path in fnames.items():
for mname in ["greedy-7b-ft-w-context-english"]:
    df[mname] = fnames[mname]
    df[f"{mname}-comet"] = comet_metric.predict(
        [
            {"mt": y, "src": x, "ref": z}
            for x, y, z in zip(
                df["source"].to_list(), fnames[mname], df["reference"].to_list()
            )
        ],
        batch_size=64,
        gpus=1,
        progress_bar=True,
        devices=[0],
    )["scores"]
df.to_csv(f"../tacl_review/bcontrast/{split}.en-{lang}.csv", index=None)

/mnt/home/sweta/envs/exec-env/lib/python3.10/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /mnt/home/sweta/envs/exec-env/lib/python3.10/site-pa ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA RTX A6000') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [1]
Predicting DataLoader 0: 100%|██████████| 33/33 [00:08<00:00,  3.87it/s]


In [5]:
import pandas as pd

split = "test"

for lang in ["de", "pt", "ko", "nl", "fr"]:
    df = pd.read_csv(f"../tacl_review/wmt24_chat/{split}.en-{lang}.csv")
    # df["lp"] = df["source_language"] + "-" + df["target_language"]

    fnames = {
        # "baseline": read_file(f"../chat-task-2024-data/{split}/en-{lang}.baseline.txt"),
        #     "gpt-4o-context": get_translations(df, "en-" + lang, "full_context_no_template", "gpt-4o", split, engiene="open-ai"),
        #     "gpt-4o-no-context": get_translations(df, "en-" + lang, "no_context_no_template", "gpt-4o", split, engiene="open-ai"),
        #     "greedy-7b-w-context": get_translations(df, "en-" + lang, "full_context", "TowerInstruct-7B-v0.2", split),
        #     "greedy-7b-wo-context": get_translations(df, "en-" + lang, "no_context", "TowerInstruct-7B-v0.2", split),
        #     "greedy-7b-5shot-w-context": get_translations(df, "en-" + lang, "full_context_5_shot", "TowerInstruct-7B-v0.2", split),
        #     "greedy-7b-5shot-wo-context": get_translations(df, "en-" + lang, "no_context_5_shot", "TowerInstruct-7B-v0.2", split),
        #    "greedy-70b-5shot-w-context": get_translations(df, "en-" + lang, "full_context_empty_sys_llama3_5_shot", "Tower-CPT-2106-WMT24-70B-SFT-Merge-greedy", split),
        #    "greedy-70b-5shot-wo-context": get_translations(df, "en-" + lang, "no_context_empty_sys_llama3_5_shot", "Tower-CPT-2106-WMT24-70B-SFT-Merge-greedy", split),
        #     "greedy-7b-ft-ind-w-context": get_translations(df, "en-" + lang, "full_context_empty_sys", "TowerInstruct-7B-w-chat-empty-sys-no-context", split),
        #     "greedy-7b-ft-ind-wo-context": get_translations(df, "en-" + lang, "no_context_empty_sys", "TowerInstruct-7B-w-chat-empty-sys-no-context", split),
        #     "greedy-7b-ft-w-context": get_translations(df, "en-" + lang, "full_context_empty_sys", "TowerInstruct-7B-w-chat-empty-sys", split),
        #     "greedy-7b-ft-wo-context": get_translations(df, "en-" + lang, "no_context_empty_sys", "TowerInstruct-7B-w-chat-empty-sys", split),
        #     "mbr-chat": read_file(f"../mbr_outputs/mbr_outputs_{split}/full_context_empty_sys/TowerInstruct-7B-w-chat-empty-sys/wmt24_chat_{split}.en-{lang}/comet_eps.out.txt"),
        #     "mbr-source-chat": read_file(f"../mbr_outputs/mbr_outputs_{split}/full_context_empty_sys/TowerInstruct-7B-w-chat-empty-sys/wmt24_chat_{split}.en-{lang}/comet_eps_context_source.out.txt"),
        # "mbr-instruct": read_file(
        #     f"../mbr_outputs/mbr_outputs_{split}/no_context/TowerInstruct-7B-v0.2/wmt24_chat_{split}.en-{lang}/comet_eps.out.txt"
        # ),
        # "mbr-source-instruct": read_file(
        #     f"../mbr_outputs/mbr_outputs_{split}/no_context/TowerInstruct-7B-v0.2/wmt24_chat_{split}.en-{lang}/comet_eps_context_source.out.txt"
        # ),
        # "towerchat-ft-7b-w-context": get_translations(
        #     df,
        #     "en-" + lang,
        #     "full_context_empty_sys",
        #     "TowerChat-1-epoch-ctx-comet-mbr-distilled",
        #     split,
        # ),
        # "towerchat-ft-7b-wo-context": get_translations(
        #     df,
        #     "en-" + lang,
        #     "no_context_empty_sys",
        #     "TowerChat-1-epoch-ctx-comet-mbr-distilled",
        #     split,
        # ),
        # "towerchat-plus-7b-w-context": get_translations(
        #     df,
        #     "en-" + lang,
        #     "full_context_empty_sys",
        #     "TowerChat-retrain-ctx-comet-mbr-distilled",
        #     split,
        # ),
        # "towerchat-plus-7b-wo-context": get_translations(
        #     df,
        #     "en-" + lang,
        #     "no_context_empty_sys",
        #     "TowerChat-retrain-ctx-comet-mbr-distilled",
        #     split,
        # ),
        "greedy-7b-ft-w-context-english": get_translations(
            df,
            "en-" + lang,
            "english_context_empty_sys",
            "TowerInstruct-7B-w-chat-empty-sys",
            split,
        ),
    }

    for mname in ["greedy-7b-ft-w-context-english"]:
    # for mname in ["towerchat-ft-7b-w-context", "towerchat-ft-7b-wo-context", "towerchat-plus-7b-w-context", "towerchat-plus-7b-wo-context"]:
        df[mname] = fnames[mname]
        df[f"{mname}-comet"] = comet_metric.predict(
            [
                {"mt": y, "src": x, "ref": z}
                for x, y, z in zip(
                    df["source"].to_list(), fnames[mname], df["reference"].to_list()
                )
            ],
            batch_size=256,
            gpus=1,
            progress_bar=True,
            devices=[0],
        )["scores"]
    df.to_csv(f"../tacl_review/wmt24_chat/{split}.en-{lang}.csv", index=None)

/mnt/home/sweta/envs/exec-env/lib/python3.10/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /mnt/home/sweta/envs/exec-env/lib/python3.10/site-pa ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA RTX A6000') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [4]
Predicting DataLoader 0: 100%|██████████| 8/8 [00:07<00:00,  1.04it/s]
/mnt/home/sweta/envs/exec-env/lib/python3.10/site-packages/lightnin

In [9]:
from typing import List


# Referred from https://github.com/amazon-science/doc-mt-metrics/blob/main/Prism/add_context.py
def add_context(
    orig_txt: List[str],
    context_same: List[str],
    context_other: List[str],
    sender_ids: List[str],
    sep_token: str = "</s>",
    ws: int = 2,
) -> List[str]:
    if not (len(orig_txt) == len(context_same) == len(context_other)):
        raise Exception(
            f"Lengths should match: len(orig_txt)={len(orig_txt)}, len(context_same)={len(context_same)}, len(context_other)={len(context_other)}"
        )
    i = 0
    augm_txt = []
    for i in range(len(orig_txt)):
        context_window = []
        for j in range(max(0, i - ws), i):
            if sender_ids[j] == sender_ids[i]:
                context_window.append(context_same[j])
            else:
                context_window.append(context_other[j])
        augm_txt.append(" {} ".format(sep_token).join(context_window + [orig_txt[i]]))
    return augm_txt


class DocCometMetric:
    def __init__(
        self, model_name="Unbabel/wmt20-comet-qe-da", batch_size=64, ref_based=True
    ):
        checkpoint_path = download_model(model_name)
        self.model = load_from_checkpoint(checkpoint_path)
        self.batch_size = batch_size
        self.model.enable_context()
        self.ref_based = ref_based

    def get_score(self, source, outputs, references=None):
        if not self.ref_based:
            del references
            return self.model.predict(
                [{"mt": y, "src": x} for x, y in zip(source, outputs)],
                batch_size=self.batch_size,
                gpus=1,
                progress_bar=True,
            )["scores"]
        else:
            return self.model.predict(
                [
                    {"mt": y, "ref": z, "src": x}
                    for x, y, z in zip(source, outputs, references)
                ],
                batch_size=self.batch_size,
                gpus=1,
                progress_bar=False,
                devices=[self.device_id],
            )["scores"]

In [10]:
context_metric = DocCometMetric(
    model_name="Unbabel/wmt20-comet-qe-da", batch_size=256, ref_based=False
)

Fetching 5 files: 100%|██████████| 5/5 [00:37<00:00,  7.53s/it]
Lightning automatically upgraded your loaded checkpoint from v1.3.5 to v2.2.3. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../../../scratch-artemis/sweta/cache/models--Unbabel--wmt20-comet-qe-da/snapshots/2e7ffc84fb67d99cf92506611766463bb9230cfb/checkpoints/model.ckpt`
/mnt/home/sweta/envs/chat-qe-env/lib/python3.10/site-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Encoder model frozen.


In [ ]:
model_names = [
    "gpt-4o-context",
    "gpt-4o-no-context",
    "greedy-7b-w-context",
    "greedy-7b-wo-context",
    "greedy-7b-5shot-w-context",
    "greedy-7b-5shot-wo-context",
    "greedy-7b-ft-w-context",
    "greedy-7b-ft-wo-context",
    "mbr",
    "mbr-source",
    "greedy-7b-ft-ind-w-context",
    "greedy-7b-ft-ind-wo-context",
    "greedy-70b-5shot-w-context",
    "greedy-70b-5shot-wo-context",
]

In [ ]:
import pandas as pd

split = "test"

for lang in ["de", "pt", "ko", "nl", "fr"]:
    df = pd.read_csv(f"../tacl_results/wmt24_chat/{split}.en-{lang}.csv")
    df.fillna("", inplace=True)
    for mname in model_names:
        doc_dfs = []
        for _, df_group in df.groupby(["doc_id"]):
            df_group["seg_id"] = list(range(len(df_group)))
            df_group[f"source_with_context"] = add_context(
                orig_txt=df_group["source"].to_list(),
                context_same=df_group["source"].to_list(),
                context_other=df_group[mname].to_list(),
                sender_ids=df_group["sender"].to_list(),
                sep_token=context_metric.model.encoder.tokenizer.sep_token,
            )
            df_group[f"mt_with_context"] = add_context(
                orig_txt=df_group[mname].to_list(),
                context_same=df_group[mname].to_list(),
                context_other=df_group["source"].to_list(),
                sender_ids=df_group["sender"].to_list(),
                sep_token=context_metric.model.encoder.tokenizer.sep_token,
            )
            doc_dfs.append(df_group)

        dfs_all = pd.concat(doc_dfs)
        df[f"{mname}-context-comet-qe"] = context_metric.get_score(
            dfs_all[f"source_with_context"], dfs_all[f"mt_with_context"]
        )
    df.to_csv(f"../tacl_results/wmt24_chat/{split}.en-{lang}.csv", index=None)

In [4]:
import datasets
from metricx23 import models
import transformers
import torch


class MetricXRef:
    def __init__(self, model_name="google/metricx-23-xl-v2p0", batch_size=1):
        self.name = "MetricXRef"
        self.batch_size = batch_size
        self.tokenizer = transformers.AutoTokenizer.from_pretrained("google/mt5-xl")
        self.model = models.MT5ForRegression.from_pretrained(model_name)
        self.model.cuda()
        self.model.eval()

    def _make_input(self, example):
        example["input"] = (
            "candidate: "
            + example["hypothesis"]
            + " reference: "
            + example["reference"]
        )
        return example

    def _tokenize(self, example):
        return self.tokenizer(
            example["input"], max_length=1024, truncation=True, padding=False
        )

    def _remove_eos(self, example):
        example["input_ids"] = example["input_ids"][:-1]
        example["attention_mask"] = example["attention_mask"][:-1]
        return example

    def _process_dataset(self, output, reference):
        ds = datasets.Dataset.from_pandas(
            pd.DataFrame(
                data=[
                    {"hypothesis": y, "reference": x} for x, y in zip(reference, output)
                ]
            )
        )
        ds = ds.map(self._make_input)
        ds = ds.map(self._tokenize)
        ds = ds.map(self._remove_eos)
        ds.set_format(
            type="torch",
            columns=["input_ids", "attention_mask"],
            device=0,
            output_all_columns=True,
        )
        return ds

    def _get_score(self, output, reference):
        ds = self._process_dataset(output, reference)

        training_args = transformers.TrainingArguments(
            per_device_eval_batch_size=self.batch_size,
            output_dir="./",
            dataloader_pin_memory=False,
        )
        with torch.no_grad():
            trainer = transformers.Trainer(
                model=self.model,
                args=training_args,
            )

            predictions, _, _ = trainer.predict(test_dataset=ds)
        return predictions

/mnt/home/sweta/envs/exec-env/lib/python3.10/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [4]:
%set_env PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python

env: PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python


In [5]:
import sys

metric = getattr(sys.modules[__name__], "MetricXRef")(
    model_name="google/metricx-23-xl-v2p0"
)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/mnt/home/sweta/envs/exec-env/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [6]:
import pandas as pd

split = "test"

# model_names = [
#     "gpt-4o-context",
#     "gpt-4o-no-context",
#     "greedy-7b-w-context",
#     "greedy-7b-wo-context",
#     "greedy-7b-ft-w-context",
#     "greedy-7b-ft-wo-context",
#     "mbr-chat",
#     "mbr-source-chat",
#     "mbr-instruct",
#     "mbr-source-instruct",
# ]


# model_names = ["towerchat-ft-7b-w-context", "towerchat-ft-7b-wo-context", "towerchat-plus-7b-w-context", "towerchat-plus-7b-wo-context"]
model_names = ["greedy-7b-ft-w-context-english"]

lang = "de"
df = pd.read_csv(f"../tacl_review/bcontrast/{split}.en-{lang}.csv")
df.fillna("", inplace=True)
for mname in model_names:
    df[f"{mname}-metricx"] = metric._get_score(
        df[mname].to_list(), df["reference"].to_list()
    )
df.to_csv(f"../tacl_review/bcontrast/{split}.en-{lang}.csv", index=None)

Map:   0%|          | 0/2100 [00:00<?, ? examples/s]

Map:   0%|          | 0/2100 [00:00<?, ? examples/s]

Map:   0%|          | 0/2100 [00:00<?, ? examples/s]

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: WARNING Path /mnt/scratch-artemis/sweta/wandb/wandb/ wasn't writable, using system temp directory.
wandb: WARNING Path /mnt/scratch-artemis/sweta/wandb/wandb/ wasn't writable, using system temp directory
wandb: Currently logged in as: sweta-agrawal. Use `wandb login --relogin` to force relogin


Error in callback <bound method _WandbInit._pause_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7f72a070aa10>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f73341a2d70, execution_count=6 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 7f73341a2200, raw_cell="import pandas as pd

split = "test"

# model_names.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B193.136.166.83/mnt/data-poseidon/sweta/context-aware-mt/chat-translation-generation/wmt24-chat-translation/notebooks/paper-eval.ipynb#X32sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


TypeError: _WandbInit._pause_backend() takes 1 positional argument but 2 were given

In [6]:
import pandas as pd

split = "test"

# model_names = [
#     "gpt-4o-context",
#     "gpt-4o-no-context",
#     "greedy-7b-w-context",
#     "greedy-7b-wo-context",
#     "greedy-7b-5shot-w-context",
#     "greedy-7b-5shot-wo-context",
#     "greedy-7b-ft-w-context",
#     "greedy-7b-ft-wo-context",
#     "mbr",
#     "mbr-source",
#     "greedy-7b-ft-ind-w-context",
#     "greedy-7b-ft-ind-wo-context",
#     "greedy-70b-5shot-w-context",
#     "greedy-70b-5shot-wo-context",
#     "mbr-instruct",
#     "mbr-source-instruct",
# ]

# model_names = ["towerchat-ft-7b-w-context", "towerchat-ft-7b-wo-context", "towerchat-plus-7b-w-context", "towerchat-plus-7b-wo-context"]
model_names = ["greedy-7b-ft-w-context-english"]

for lang in ["de", "pt", "ko", "nl", "fr"]:
    df = pd.read_csv(f"../tacl_review/wmt24_chat/{split}.en-{lang}.csv")
    df.fillna("", inplace=True)
    for mname in model_names:
        df[f"{mname}-metricx"] = metric._get_score(
            df[mname].to_list(), df["reference"].to_list()
        )
    df.to_csv(f"../tacl_review/wmt24_chat/{split}.en-{lang}.csv", index=None)

Map:   0%|          | 0/2041 [00:00<?, ? examples/s]

Map:   0%|          | 0/2041 [00:00<?, ? examples/s]

Map:   0%|          | 0/2041 [00:00<?, ? examples/s]

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: sweta-agrawal. Use `wandb login --relogin` to force relogin


...

Map:   0%|          | 0/2091 [00:00<?, ? examples/s]

Map:   0%|          | 0/2091 [00:00<?, ? examples/s]

Map:   0%|          | 0/2091 [00:00<?, ? examples/s]

Error in callback <bound method _WandbInit._pause_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7f40eb9dcb50>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f3f19ad38e0, execution_count=6 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 7f3f19ad3520, raw_cell="import pandas as pd

split = "test"

# model_names.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B193.136.166.83/mnt/data-poseidon/sweta/context-aware-mt/chat-translation-generation/wmt24-chat-translation/notebooks/paper-eval.ipynb#X33sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


TypeError: _WandbInit._pause_backend() takes 1 positional argument but 2 were given

In [4]:
import sacrebleu
import pandas as pd

split = "test"

model_names = [
    "gpt-4o-context",
    "gpt-4o-no-context",
    "greedy-7b-w-context",
    "greedy-7b-wo-context",
    "greedy-7b-ft-w-context",
    "greedy-7b-ft-wo-context",
    "mbr",
    "mbr-source",
    "mbr-instruct",
    "mbr-source-instruct",
    "towerchat-ft-7b-w-context", 
    "towerchat-ft-7b-wo-context", 
    "towerchat-plus-7b-w-context", 
    "towerchat-plus-7b-wo-context",
    "greedy-7b-ft-w-context-english"
]


all_scores = []
for lang in ["de", "fr", "pt", "ko", "nl"]:
    df = pd.read_csv(f"../tacl_review/wmt24_chat/{split}.en-{lang}.csv")
    df.fillna("", inplace=True)
    # for mname in model_names:
    #     all_scores.append([lang, mname, sacrebleu.corpus_chrf(df[mname].to_list(), [df["reference"].to_list()]).score, df[f"{mname}-comet"].mean(), df[f"{mname}-metricx"].mean(), df[f"{mname}-context-comet-qe"].mean(), ])

    for lp, lp_df in df.groupby("lp"):
        for mname in model_names:
            all_scores.append(
                [
                    lp,
                    mname,
                    sacrebleu.corpus_chrf(
                        lp_df[mname].to_list(), [lp_df["reference"].to_list()]
                    ).score,
                    lp_df[f"{mname}-comet"].mean(),
                    lp_df[f"{mname}-metricx"].mean(),
                ]
            )

In [11]:
import sacrebleu
import pandas as pd

split = "test"

model_names = [
    "gpt-4o-context",
    "gpt-4o-no-context",
    "greedy-7b-w-context",
    "greedy-7b-wo-context",
    "greedy-7b-ft-w-context",
    "greedy-7b-ft-wo-context",
    "mbr-chat",
    "mbr-source-chat",
    "mbr-instruct",
    "mbr-source-instruct",
    "towerchat-ft-7b-w-context", 
    "towerchat-ft-7b-wo-context", 
    "towerchat-plus-7b-w-context", 
    "towerchat-plus-7b-wo-context",
    "greedy-7b-ft-w-context-english"
]

all_scores = []
lang = "de"
df = pd.read_csv(f"../tacl_review/bcontrast/{split}.en-{lang}.csv")
df.fillna("", inplace=True)
for lp, lp_df in df.groupby("lp"):
    for mname in model_names:
        all_scores.append(
            [
                lp,
                mname,
                sacrebleu.corpus_chrf(
                    lp_df[mname].to_list(), [lp_df["reference"].to_list()]
                ).score,
                lp_df[f"{mname}-comet"].mean(),
                lp_df[f"{mname}-metricx"].mean(),
            ]
        )

In [12]:
scores_df = pd.DataFrame(all_scores)
# scores_df.columns = ["lp", "model", "chrf", "comet", "metricx", "comet-qe"]
scores_df.columns = ["lp", "model", "chrf", "comet", "metricx"]

scores_df["comet"] = scores_df["comet"] * 100
# scores_df["comet-qe"] = scores_df["comet-qe"]*100

In [6]:
for gr, gr_df in scores_df.groupby("model"):
    enxx = (" & ").join(
        [
            f"{gr_df[gr_df.lp=='en-'+lang]['comet'].values[0]:.2f}"
            for lang in ["de", "fr", "pt-br", "ko", "nl"]
        ]
    )
    xxen = (" & ").join(
        [
            f"{gr_df[gr_df.lp==lang + '-en']['comet'].values[0]:.2f}"
            for lang in ["de", "fr", "pt-br", "ko", "nl"]
        ]
    )
    print(f"{gr} & {enxx} && {xxen} ")

gpt-4o-context & 92.49 & 92.62 & 93.40 & 93.06 & 93.08 && 91.95 & 91.76 & 91.19 & 90.67 & 92.39 
gpt-4o-no-context & 92.74 & 92.43 & 93.01 & 92.26 & 92.68 && 92.16 & 92.18 & 91.40 & 93.24 & 93.07 
greedy-7b-ft-w-context & 92.74 & 92.64 & 94.53 & 94.13 & 94.09 && 92.24 & 92.67 & 92.09 & 94.98 & 93.06 
greedy-7b-ft-w-context-english & 92.94 & 92.70 & 94.38 & 94.10 & 92.51 && 92.30 & 92.68 & 91.67 & 94.87 & 92.98 
greedy-7b-ft-wo-context & 92.36 & 92.26 & 93.89 & 93.73 & 92.81 && 92.28 & 92.79 & 91.06 & 94.69 & 92.78 
greedy-7b-w-context & 91.48 & 91.08 & 90.79 & 91.13 & 91.00 && 91.33 & 91.89 & 90.63 & 91.88 & 91.08 
greedy-7b-wo-context & 91.71 & 91.89 & 91.90 & 91.64 & 91.30 && 92.08 & 92.78 & 90.43 & 93.13 & 92.45 
mbr & 93.28 & 93.13 & 94.91 & 95.01 & 94.54 && 92.58 & 92.95 & 92.63 & 95.32 & 93.49 
mbr-instruct & 92.77 & 92.62 & 93.24 & 93.04 & 92.68 && 92.78 & 93.34 & 91.13 & 93.87 & 92.88 
mbr-source & 93.22 & 92.96 & 94.76 & 94.96 & 94.36 && 92.48 & 92.71 & 92.46 & 95.16 & 93.38 


In [7]:
for gr, gr_df in scores_df.groupby("model"):
    enxx = (" & ").join(
        [
            f"{gr_df[gr_df.lp=='en-'+lang]['metricx'].values[0]:.2f}"
            for lang in ["de", "fr", "pt-br", "ko", "nl"]
        ]
    )
    xxen = (" & ").join(
        [
            f"{gr_df[gr_df.lp==lang + '-en']['metricx'].values[0]:.2f}"
            for lang in ["de", "fr", "pt-br", "ko", "nl"]
        ]
    )
    print(f"{gr} & {enxx} && {xxen} ")

gpt-4o-context & 0.45 & 0.22 & 0.28 & 0.39 & 0.30 && 0.55 & 0.56 & 0.64 & 0.65 & 0.50 
gpt-4o-no-context & 0.42 & 0.28 & 0.29 & 0.51 & 0.33 && 0.52 & 0.51 & 0.67 & 0.32 & 0.46 
greedy-7b-ft-w-context & 0.34 & 0.26 & 0.27 & 0.45 & 0.27 && 0.47 & 0.48 & 0.60 & 0.30 & 0.48 
greedy-7b-ft-w-context-english & 0.26 & 0.24 & 0.28 & 0.46 & 0.41 && 0.49 & 0.47 & 0.60 & 0.30 & 0.49 
greedy-7b-ft-wo-context & 0.27 & 0.24 & 0.29 & 0.42 & 0.37 && 0.50 & 0.51 & 0.71 & 0.33 & 0.52 
greedy-7b-w-context & 0.38 & 0.29 & 0.69 & 0.60 & 0.49 && 0.56 & 0.55 & 0.74 & 0.46 & 0.69 
greedy-7b-wo-context & 0.28 & 0.23 & 0.43 & 0.57 & 0.37 && 0.50 & 0.53 & 0.86 & 0.37 & 0.53 
mbr & 0.30 & 0.22 & 0.24 & 0.31 & 0.21 && 0.46 & 0.46 & 0.55 & 0.27 & 0.45 
mbr-instruct & 0.25 & 0.20 & 0.37 & 0.42 & 0.31 && 0.48 & 0.49 & 0.78 & 0.35 & 0.50 
mbr-source & 0.31 & 0.22 & 0.24 & 0.29 & 0.23 && 0.47 & 0.47 & 0.56 & 0.27 & 0.45 
mbr-source-instruct & 0.26 & 0.19 & 0.33 & 0.43 & 0.31 && 0.47 & 0.48 & 0.79 & 0.33 & 0.50 
towercha

In [8]:
for gr, gr_df in scores_df.groupby("model"):
    enxx = (" & ").join(
        [
            f"{gr_df[gr_df.lp=='en-'+lang]['chrf'].values[0]:.2f}"
            for lang in ["de", "fr", "pt-br", "ko", "nl"]
        ]
    )
    xxen = (" & ").join(
        [
            f"{gr_df[gr_df.lp==lang + '-en']['chrf'].values[0]:.2f}"
            for lang in ["de", "fr", "pt-br", "ko", "nl"]
        ]
    )
    print(f"{gr} & {enxx} && {xxen} ")

gpt-4o-context & 74.96 & 78.45 & 76.05 & 48.78 & 73.49 && 78.01 & 77.64 & 72.58 & 69.21 & 76.28 
gpt-4o-no-context & 76.67 & 78.69 & 75.81 & 47.63 & 71.65 && 79.60 & 78.27 & 73.73 & 76.45 & 78.62 
greedy-7b-ft-w-context & 76.41 & 79.97 & 82.24 & 61.28 & 79.78 && 79.91 & 79.26 & 75.72 & 81.30 & 78.15 
greedy-7b-ft-w-context-english & 77.31 & 80.27 & 81.67 & 61.06 & 73.73 && 79.87 & 79.14 & 74.97 & 81.52 & 78.31 
greedy-7b-ft-wo-context & 74.04 & 77.12 & 79.71 & 57.63 & 69.91 && 79.31 & 79.36 & 74.00 & 80.17 & 77.01 
greedy-7b-w-context & 71.16 & 74.38 & 68.50 & 41.70 & 61.23 && 75.68 & 78.31 & 71.83 & 72.63 & 73.15 
greedy-7b-wo-context & 71.81 & 74.59 & 72.26 & 43.18 & 62.90 && 77.57 & 79.02 & 72.06 & 75.73 & 75.80 
mbr & 77.09 & 80.34 & 82.25 & 61.79 & 80.33 && 79.70 & 78.78 & 75.88 & 81.56 & 78.67 
mbr-instruct & 72.74 & 73.60 & 72.95 & 43.11 & 63.59 && 76.26 & 79.44 & 71.86 & 75.32 & 75.07 
mbr-source & 77.23 & 80.51 & 82.55 & 62.29 & 80.25 && 79.87 & 78.57 & 76.01 & 81.57 & 78.60 


In [10]:
scores_df["type"] = scores_df["lp"].apply(
    lambda x: "en-xx" if x.split("-")[0] == "en" else "xx-en"
)
for gr, gr_df in scores_df.groupby("model"):
    print(
        f"{gr} & {gr_df[gr_df.type=='en-xx']['chrf'].mean():.2f} & {gr_df[gr_df.type=='en-xx']['comet'].mean():.2f} & {gr_df[gr_df.type=='en-xx']['metricx'].mean():.2f}"
        " && "
        f"{gr_df[gr_df.type=='xx-en']['chrf'].mean():.2f} & {gr_df[gr_df.type=='xx-en']['comet'].mean():.2f}& {gr_df[gr_df.type=='xx-en']['metricx'].mean():.2f}"
    )

gpt-4o-context & 70.34 & 92.93 & 0.33 && 74.75 & 91.59& 0.58
gpt-4o-no-context & 70.09 & 92.62 & 0.37 && 77.33 & 92.41& 0.50
greedy-7b-ft-w-context & 75.94 & 93.63 & 0.32 && 78.87 & 93.01& 0.47
greedy-7b-ft-w-context-english & 74.81 & 93.33 & 0.33 && 78.76 & 92.90& 0.47
greedy-7b-ft-wo-context & 71.68 & 93.01 & 0.32 && 77.97 & 92.72& 0.51
greedy-7b-w-context & 63.39 & 91.09 & 0.49 && 74.32 & 91.36& 0.60
greedy-7b-wo-context & 64.95 & 91.69 & 0.38 && 76.04 & 92.17& 0.56
mbr & 76.36 & 94.18 & 0.25 && 78.92 & 93.39& 0.44
mbr-instruct & 65.20 & 92.87 & 0.31 && 75.59 & 92.80& 0.52
mbr-source & 76.56 & 94.05 & 0.26 && 78.92 & 93.24& 0.44
mbr-source-instruct & 65.06 & 92.57 & 0.31 && 75.91 & 92.65& 0.51
towerchat-ft-7b-w-context & 76.22 & 93.72 & 0.29 && 78.80 & 93.04& 0.47
towerchat-ft-7b-wo-context & 73.04 & 93.25 & 0.31 && 78.40 & 92.81& 0.50
towerchat-plus-7b-w-context & 75.51 & 93.60 & 0.29 && 78.20 & 93.01& 0.48
towerchat-plus-7b-wo-context & 71.50 & 93.10 & 0.31 && 77.93 & 92.77& 0.51


In [ ]:
#bcontrast
scores_df["type"] = scores_df["lp"].apply(
    lambda x: "en-xx" if x.split("-")[0] == "en" else "xx-en"
)
for gr, gr_df in scores_df.groupby("model"):
    print(
        f"{gr} & {gr_df[gr_df.type=='en-xx']['chrf'].mean():.2f} & {gr_df[gr_df.type=='en-xx']['comet'].mean():.2f} & {gr_df[gr_df.type=='en-xx']['metricx'].mean():.2f}"
        " && "
        f"{gr_df[gr_df.type=='xx-en']['chrf'].mean():.2f} & {gr_df[gr_df.type=='xx-en']['comet'].mean():.2f}& {gr_df[gr_df.type=='xx-en']['metricx'].mean():.2f}"
    )

gpt-4o-context & 70.23 & 90.96 & 0.39 && 72.72 & 92.81& 0.39
gpt-4o-no-context & 68.51 & 90.60 & 0.52 && 71.14 & 92.35& 0.41
greedy-7b-ft-w-context & 67.09 & 89.60 & 0.46 && 69.31 & 92.05& 0.44
greedy-7b-ft-w-context-english & 66.38 & 89.53 & 0.47 && 69.33 & 92.05& 0.44
greedy-7b-ft-wo-context & 65.32 & 89.45 & 0.49 && 68.74 & 91.74& 0.48
greedy-7b-w-context & 62.75 & 88.29 & 0.56 && 70.35 & 91.99& 0.46
greedy-7b-wo-context & 62.46 & 88.20 & 0.54 && 69.54 & 91.98& 0.47
mbr-chat & 67.84 & 90.94 & 0.40 && 69.71 & 92.34& 0.43
mbr-instruct & 63.42 & 90.15 & 0.46 && 69.24 & 92.31& 0.44
mbr-source-chat & 67.69 & 90.58 & 0.40 && 69.65 & 92.24& 0.43
mbr-source-instruct & 64.10 & 89.78 & 0.46 && 69.39 & 92.22& 0.44
towerchat-ft-7b-w-context & 67.34 & 89.81 & 0.45 && 69.16 & 91.87& 0.44
towerchat-ft-7b-wo-context & 65.92 & 89.70 & 0.46 && 69.02 & 91.85& 0.46
towerchat-plus-7b-w-context & 66.49 & 89.57 & 0.47 && 68.79 & 91.92& 0.44
towerchat-plus-7b-wo-context & 64.82 & 89.17 & 0.49 && 68.37 & 91

In [2]:
import json
import pandas as pd

split = "test"


def load_jsonl_file(file_path):
    with open(file_path, "r") as f:
        json_list = list(f)
    return [json.loads(l) for l in json_list]


def get_df(model_names, langs, data_name):
    df_dict = {
        "model": [],
        "lp": [],
        "context": [],
        "tag": [],
        "precision": [],
        "recall": [],
        "f1": [],
    }

    data = []

    for l in langs:
        for model_name in model_names:
            stats = load_jsonl_file(
                f"../tacl_review/{data_name}/{split}_{l}_{model_name}_stats.jsonl"
            )
            for s in stats:
                for metric in ["precision", "recall", "f1"]:
                    data.append(
                        {
                            "model": model_name,
                            "lp": f"en-{l}",
                            "tag": s["tag"],
                            "metric": metric,
                            "value": s[metric],
                        }
                    )

    df = pd.DataFrame().from_dict(data)
    return df

In [5]:
import numpy as np
langs = ["de", "fr", "pt", "nl", "ko"]
data_name = "wmt24_chat"

model_names = [
    # "greedy-7b-w-context",
    # "greedy-7b-ft-w-context",
    # "greedy-7b-wo-context",
    # "greedy-7b-ft-wo-context",
    # "mbr",
    # "mbr-source",
    # "mbr-instruct",
    # "mbr-source-instruct",
    "towerchat-ft-7b-w-context",
    "towerchat-ft-7b-wo-context",
]
wmt24_chat_df = get_df(model_names, langs, data_name)

for gr, gr_df in wmt24_chat_df.groupby("model"):
    scores = [
        f"{gr_df[(gr_df.tag==tag) &  (gr_df.metric=='precision')]['value'].mean() * 100:.2f}"
        for tag in ["lexical_cohesion", "verb_form", "pronouns", "formality"]
    ]
    print(f"{gr} & {(' & ').join(scores)} & {np.mean(list(map(float, scores))):.2f} ")

towerchat-ft-7b-w-context & 86.36 & 61.15 & 81.52 & 87.20 & 79.06 
towerchat-ft-7b-wo-context & 85.73 & 64.59 & 78.86 & 68.63 & 74.45 


In [9]:
langs = ["de"]
data_name = "bcontrast"

model_names = [
    # "greedy-7b-w-context",
    # "greedy-7b-ft-w-context",
    # "greedy-7b-wo-context",
    # "greedy-7b-ft-wo-context",
    # "mbr-chat",
    # "mbr-source-chat",
    # "mbr-instruct",
    # "mbr-source-instruct",
    "towerchat-ft-7b-w-context",
    "towerchat-ft-7b-wo-context",
]
bcontrast_df = get_df(model_names, langs, data_name)

import numpy as np

for gr, gr_df in bcontrast_df.groupby("model"):
    scores = [
        f"{gr_df[(gr_df.tag==tag) &  (gr_df.metric=='precision')]['value'].mean() * 100:.2f}"
        for tag in ["lexical_cohesion", "pronouns", "formality"]
    ]
    print(f"{gr} & {(' & ').join(scores)} & {np.mean(list(map(float, scores))):.2f} ")

towerchat-ft-7b-w-context & 40.00 & 61.19 & 86.68 & 62.62 
towerchat-ft-7b-wo-context & 62.26 & 52.38 & 86.06 & 66.90 


# MuDA fix results

In [1]:
import os

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"  # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"] = "4"

import sys
import numpy as np

sys.path.append(
    "/mnt/data-poseidon/sweta/context-aware-mt/chat-translation-generation/wmt24-chat-translation/metrics/MuDA"
)

from nltk.corpus import stopwords

nltk_stopwords = set(stopwords.words("english"))
conversational_stopwords = {"um", "uh", "okay", "ok", "yes", "no"}

stop_words = nltk_stopwords.union(conversational_stopwords)

In [2]:
stop_words = conversational_stopwords

In [3]:
from typing import List, Dict, Tuple

from collections import defaultdict
from itertools import chain

from muda.tagger import Tagger, Tagging


def compute_metrics(
    tagged_refs: List[List[List[Tagging]]], tagged_hyps: List[List[List[Tagging]]]
) -> Tuple[Dict[str, float], Dict[str, float], Dict[str, float]]:
    """Computes the accuracy, recall and f1 for each tag based if word tagged in the
    reference/hypothesis exist and are also tagged in the hypothesis/reference.

    Inspired by the compare-mt's LabelWordBucketer:
    https://github.com/neulab/compare-mt/blob/master/compare_mt/bucketers.py"""
    flatten_refs = chain.from_iterable(tagged_refs)
    flatten_hyps = chain.from_iterable(tagged_hyps)

    tagref_matches: Dict[str, int] = defaultdict(int)
    tagref_total: Dict[str, int] = defaultdict(int)
    taghyp_matches: Dict[str, int] = defaultdict(int)
    taghyp_total: Dict[str, int] = defaultdict(int)

    for tgt_sent, hyp_sent in zip(flatten_refs, flatten_hyps):
        # get position of words in the reference
        # and mark the tags that appear in the reference
        # print(tgt_sent, hyp_sent)
        ref_pos = defaultdict(list)
        for i, tagging in enumerate(tgt_sent):
            word = Tagger.normalize(tagging["token"]).lower()
            # print(word)
            if word in stop_words:
                continue
            ref_pos[word].append(i)
            for tag in tagging["tags"]:
                tagref_total[tag] += 1

        word_count: Dict[str, int] = defaultdict(int)
        for i, tagging in enumerate(hyp_sent):
            word = Tagger.normalize(tagging["token"]).lower()
            if word in stop_words:
                continue
            for tag in tagging["tags"]:
                taghyp_total[tag] += 1

            if word in ref_pos and word_count[word] < len(ref_pos[word]):
                # mark intersection of tags in the reference and hypothesis
                for tag in tagging["tags"]:
                    if tag in tgt_sent[ref_pos[word][word_count[word]]]["tags"]:
                        tagref_matches[tag] += 1
                        taghyp_matches[tag] += 1

            word_count[word] += 1

    prec: Dict[str, float] = defaultdict(float)
    rec: Dict[str, float] = defaultdict(float)
    prec.update({tag: taghyp_matches[tag] / taghyp_total[tag] for tag in taghyp_total})
    rec.update({tag: tagref_matches[tag] / tagref_total[tag] for tag in tagref_total})
    all_tags = set(tagref_total.keys()).union(set(taghyp_total.keys()))
    f1 = {
        tag: 2 * prec[tag] * rec[tag] / max(prec[tag] + rec[tag], 1e-20)
        for tag in all_tags
    }
    return prec, rec, f1

/mnt/home/sweta/envs/chat-qe-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-02-19 22:38:15.797167: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-02-19 22:38:17.412990: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-02-19 22:38:27.741495: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could n

In [17]:
import json
import pandas as pd

split = "test"


def load_jsonl_file(file_path):
    with open(file_path, "r") as f:
        json_list = list(f)
    return [json.loads(l) for l in json_list]


def get_custom_df(model_names, langs, data_name):

    data = []

    for l in langs:
        for model_name in model_names:
            if "towerchat-ft-7b" in model_name:
                tagged_hyps = json.load(
                    open(
                        f"../tacl_review/{data_name}/{split}_{l}_{model_name}_hyps_tags.json"
                    )
                )
                tagged_refs = json.load(
                    open(
                        f"../tacl_review/{data_name}/{split}_{l}_{model_name}_refs_tags.json"
                    )
                )
            else:
                tagged_hyps = json.load(
                    open(
                        f"../tacl_results/{data_name}/{split}_{l}_{model_name}_hyps_tags.json"
                    )
                )
                tagged_refs = json.load(
                    open(
                        f"../tacl_results/{data_name}/{split}_{l}_{model_name}_refs_tags.json"
                    )
                )

            tag_prec, tag_rec, tag_f1 = compute_metrics(tagged_refs, tagged_hyps)
            for tag in tag_f1:
                data.append(
                    {
                        "model": model_name,
                        "lp": f"en-{l}",
                        "tag": tag,
                        "metric": "f1",
                        "value": tag_f1[tag],
                    }
                )
                data.append(
                    {
                        "model": model_name,
                        "lp": f"en-{l}",
                        "tag": tag,
                        "metric": "recall",
                        "value": tag_rec[tag],
                    }
                )
                data.append(
                    {
                        "model": model_name,
                        "lp": f"en-{l}",
                        "tag": tag,
                        "metric": "precision",
                        "value": tag_prec[tag],
                    }
                )
    df = pd.DataFrame().from_dict(data)
    return df

In [18]:
langs = ["de", "fr", "pt", "nl", "ko"]
data_name = "wmt24_chat"

model_names = [
    "greedy-7b-wo-context",
    "greedy-7b-w-context",
    "mbr-instruct",
    "mbr-source-instruct",
    "greedy-7b-ft-wo-context",
    "greedy-7b-ft-w-context",
    "mbr",
    "mbr-source",
    "towerchat-ft-7b-w-context",
    "towerchat-ft-7b-wo-context",
]
wmt24_chat_df = get_custom_df(model_names, langs, data_name)
wmt24_chat_df["dataset"] = "WMT24"
tag_metric = {
    "lexical_cohesion": "f1",
    "verb_form": "f1",
    "pronouns": "f1",
    "formality": "f1",
}

In [19]:
langs = ["de"]
data_name = "bcontrast"

model_names = [
    "greedy-7b-wo-context",
    "greedy-7b-w-context",
    "mbr-instruct",
    "mbr-source-instruct",
    "greedy-7b-ft-wo-context",
    "greedy-7b-ft-w-context",
    "mbr-chat",
    "mbr-source-chat",
    "towerchat-ft-7b-w-context",
    "towerchat-ft-7b-wo-context",
]
bcontrast_df = get_custom_df(model_names, langs, data_name)

bcontrast_df["dataset"] = "BCONTRAST"
tag_metric = {"lexical_cohesion": "f1", "pronouns": "f1", "formality": "f1"}

In [20]:
df = pd.concat([bcontrast_df, wmt24_chat_df], axis=0)

In [21]:
df["lp"] = df["lp"].apply(lambda x: x.upper())

In [22]:
df["dataset_lp"] = df["dataset"] + " " + df["lp"]

In [24]:
model_map = {
    "greedy-7b-ft-w-context": "\\TowerChat{}  &  \\cmark",
    "greedy-7b-w-context": "\\TowerInstruct{}   & \\cmark  ",
    "greedy-7b-ft-wo-context": "\\TowerChat{}  & \\xmark ",
    "greedy-7b-wo-context": "\\TowerInstruct{}  & \\xmark ",
    "mbr-instruct": "\\TowerInstruct{} + \\quad + QAD (\\comet{}) & \\xmark",
    "mbr-source-instruct": "\\TowerInstruct{} + \\quad + QAD (\\contextcomet{}) & \\xmark",
    "mbr": "\\TowerChat{} + \\quad + QAD (\\comet{}) & \\cmark",
    "mbr-source": "\\TowerChat{} + \\quad + QAD (\\contextcomet{}) & \\cmark",
    "mbr-chat": "\\TowerChat{} + \\quad + QAD (\\comet{}) & \\cmark",
    "mbr-source-chat": "\\TowerChat{} + \\quad + QAD (\\contextcomet{}) & \\cmark",
    "towerchat-ft-7b-w-context": "\\TowerChat{} + \\quad + QAD Distilliation & \\cmark",
    "towerchat-ft-7b-wo-context": "\\TowerChat{} + \\quad + QAD Distilliation & \\xmark",
}

df["model"] = df["model"].apply(lambda x: model_map[x])

In [25]:
tag_map = {
    "verb_form": "Verb Form",
    "pronouns": "Pronouns",
    "lexical_cohesion": "Lexical Cohesion",
    "formality": "Formality",
}

df["tag"] = df["tag"].apply(lambda x: tag_map[x])

In [27]:
for model in df.model.unique():
    scores = {}
    for gr, gr_df in df.groupby("dataset_lp"):
        scores_lp = [
            f"{gr_df[(gr_df.model==model) & (gr_df.tag==tag) &  (gr_df.metric=='f1')]['value'].mean() * 100:.2f}"
            for tag in ["Lexical Cohesion", "Verb Form", "Pronouns", "Formality"]
        ]
        scores[gr] = np.mean(list(map(float, [x for x in scores_lp if x != "nan"])))

    scores_ordered = [
        f"{scores[x]:.2f}"
        for x in [
            "WMT24 EN-DE",
            "WMT24 EN-PT",
            "WMT24 EN-FR",
            "WMT24 EN-NL",
            "WMT24 EN-KO",
            "BCONTRAST EN-DE",
        ]
    ]
    print(f"{model} & {(' & & ').join(scores_ordered)} \\\\")

\TowerInstruct{}  & \xmark  & 78.29 & & 76.56 & & 80.44 & & 52.40 & & 52.44 & & 69.47 \\
\TowerInstruct{}   & \cmark   & 80.12 & & 73.44 & & 80.23 & & 46.46 & & 49.06 & & 69.32 \\
\TowerInstruct{} + \quad + QAD (\comet{}) & \xmark & 78.39 & & 77.64 & & 80.47 & & 54.80 & & 54.77 & & 71.63 \\
\TowerInstruct{} + \quad + QAD (\contextcomet{}) & \xmark & 78.42 & & 76.47 & & 80.62 & & 52.57 & & 55.62 & & 69.76 \\
\TowerChat{}  & \xmark  & 77.90 & & 79.53 & & 82.99 & & 49.69 & & 60.53 & & 68.16 \\
\TowerChat{}  &  \cmark & 79.81 & & 86.55 & & 86.34 & & 68.27 & & 60.12 & & 76.92 \\
\TowerChat{} + \quad + QAD (\comet{}) & \cmark & 80.02 & & 87.24 & & 87.11 & & 76.96 & & 62.50 & & 76.56 \\
\TowerChat{} + \quad + QAD (\contextcomet{}) & \cmark & 80.68 & & 85.94 & & 87.17 & & 78.74 & & 64.34 & & 74.50 \\
\TowerChat{} + \quad + QAD Distilliation & \cmark & 84.11 & & 87.49 & & 87.41 & & 71.07 & & 63.94 & & 76.12 \\
\TowerChat{} + \quad + QAD Distilliation & \xmark & 81.01 & & 84.28 & & 85.80 & & 52.

In [14]:
for gr, gr_df in df.groupby("dataset_lp"):
    print(gr)
    for model in gr_df.model.unique():
        scores = [
            f"{gr_df[(gr_df.model==model) & (gr_df.tag==tag) &  (gr_df.metric=='f1')]['value'].mean() * 100:.2f}"
            for tag in ["Lexical Cohesion", "Verb Form", "Pronouns", "Formality"]
        ]
        print(
            f"{model} & {(' & ').join(scores).replace('nan', '-')} & {np.mean(list(map(float, [x for x in scores if x!= 'nan']))):.2f} \\\\"
        )

BCONTRAST EN-DE
\TowerChat{} + \quad + QAD Distilliation & \cmark & 75.00 & - & 66.67 & 86.68 & 76.12 \\
\TowerChat{} + \quad + QAD Distilliation & \xmark & 73.08 & - & 55.46 & 84.11 & 70.88 \\
WMT24 EN-DE
\TowerChat{} + \quad + QAD Distilliation & \cmark & 87.34 & - & 73.85 & 91.15 & 84.11 \\
\TowerChat{} + \quad + QAD Distilliation & \xmark & 85.20 & - & 67.19 & 90.65 & 81.01 \\
WMT24 EN-FR
\TowerChat{} + \quad + QAD Distilliation & \cmark & 94.37 & 71.30 & 92.52 & 91.47 & 87.41 \\
\TowerChat{} + \quad + QAD Distilliation & \xmark & 90.84 & 71.54 & 90.79 & 90.02 & 85.80 \\
WMT24 EN-KO
\TowerChat{} + \quad + QAD Distilliation & \cmark & 71.81 & - & - & 56.06 & 63.94 \\
\TowerChat{} + \quad + QAD Distilliation & \xmark & 69.84 & - & - & 51.37 & 60.61 \\
WMT24 EN-NL
\TowerChat{} + \quad + QAD Distilliation & \cmark & 88.00 & 31.58 & - & 93.64 & 71.07 \\
\TowerChat{} + \quad + QAD Distilliation & \xmark & 85.71 & 45.45 & - & 26.05 & 52.40 \\
WMT24 EN-PT
\TowerChat{} + \quad + QAD Distill

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("ticks")
sns.set_context("poster")

# activate tex
plt.rc("text", usetex=False)
sns.set_context("paper", font_scale=2.5)

In [35]:
from matplotlib.backends.backend_pdf import PdfPages

In [ ]:
other_models = df["model"].unique().tolist()

palette = sns.color_palette("colorblind", len(other_models))

model_palette = {model: color for model, color in zip(other_models, palette)}
# model_palette['TowerInstruct w/o context (Greedy)'] = 'grey'  # Set 'TowerInstruct w/o context (Greedy)' to grey
# model_order = ['TowerInstruct w/o context (Greedy)'] + other_models
model_order = other_models

with sns.axes_style("darkgrid"):
    fig, axs = plt.subplots(
        len(df["dataset_lp"].unique()),
        1,
        figsize=(12, 8 * len(df["lp"].unique())),
        sharex=True,
    )
    for i, lp in enumerate(df["dataset_lp"].unique()):
        # Filter the dataframe for the current lp
        df_lp = df[df["dataset_lp"] == lp]

        # Create a barplot for the current lp
        sns.barplot(
            data=df_lp,
            x="tag",
            y="value",
            hue="model",
            palette=model_palette,
            hue_order=model_order,
            ax=axs[i],
            legend=(i == 0),
        )

        # Add vertical lines between x-axis categories
        for j in range(
            len(df[df["dataset_lp"] == "BCONTRAST EN-DE"]["tag"].unique()) - 1
        ):
            axs[i].axvline(x=j + 0.5, color="black", linestyle="--", linewidth=0.8)

        # Set the plot title and labels
        axs[i].set_title(f"{lp.upper()}")
        axs[i].set_ylim(0, 1)  # Set y-axis limits between 0 and 1
        axs[i].set_ylabel("F1 Score")
        axs[i].set_xlabel("")
        if i == 0:
            handles, labels = axs[
                0
            ].get_legend_handles_labels()  # Get handles and labels
            axs[i].legend(
                handles,
                labels,
                loc="upper center",
                bbox_to_anchor=(0.5, 1.5),
                ncol=2,
                fontsize=16,
            )  # Place legend above the plot


# plt.show()
with PdfPages("../plots/all_barplots_muda.pdf") as pdf:
    pdf.savefig(fig, bbox_inches="tight")

# Number of turns

In [ ]:
import pandas as pd

split = "test"

for lang in ["de", "pt", "ko", "nl", "fr"]:
    df = pd.read_csv(f"../tacl_results/wmt24_chat/{split}.en-{lang}.csv")
    df["lp"] = df["source_language"] + "-" + df["target_language"]

    fnames = {}
    for turn in [
        "no_context_empty_sys",
        "full_context_empty_sys_2_turns",
        "full_context_empty_sys_6_turns",
        "full_context_empty_sys_10_turns",
        "full_context_empty_sys_15_turns",
        "full_context_empty_sys_20_turns",
        "full_context_empty_sys",
    ]:
        fnames[turn] = get_translations(
            df, "en-" + lang, turn, "TowerInstruct-7B-w-chat-empty-sys", split
        )

    for mname, file_path in fnames.items():
        df[mname] = fnames[mname]
        df[f"{mname}-comet"] = comet_metric.predict(
            [
                {"mt": y, "src": x, "ref": z}
                for x, y, z in zip(
                    df["source"].to_list(), fnames[mname], df["reference"].to_list()
                )
            ],
            batch_size=256,
            gpus=1,
            progress_bar=True,
            devices=[0],
        )["scores"]
    df.to_csv(f"../tacl_results/wmt24_chat/{split}.en-{lang}.turns.csv", index=None)

In [38]:
import sacrebleu
import pandas as pd

split = "test"

model_names = {
    0: "no_context_empty_sys",
    2: "full_context_empty_sys_2_turns",
    6: "full_context_empty_sys_6_turns",
    10: "full_context_empty_sys_10_turns",
    15: "full_context_empty_sys_15_turns",
    20: "full_context_empty_sys_20_turns",
    100: "full_context_empty_sys",
}

all_scores = []
for lang in ["de", "fr", "pt", "ko", "nl"]:
    df = pd.read_csv(f"../tacl_results/wmt24_chat/{split}.en-{lang}.turns.csv")
    df.fillna("", inplace=True)

    for lp, lp_df in df.groupby("lp"):
        for t, mname in model_names.items():
            all_scores.append(
                [
                    lp,
                    t,
                    sacrebleu.corpus_chrf(
                        lp_df[mname].to_list(), [lp_df["reference"].to_list()]
                    ).score,
                    lp_df[f"{mname}-comet"].mean(),
                ]
            )

scores_df = pd.DataFrame(all_scores)
scores_df.columns = ["LP", "# of Turns", "CHRF", "COMET"]

scores_df["COMET"] = scores_df["COMET"] * 100

scores_df["# of Turns"] = scores_df["# of Turns"].astype(str)
scores_df["# of Turns"] = scores_df["# of Turns"].replace("100", "Full")
scores_df["# of Turns"] = scores_df["# of Turns"].replace("0", "No")
scores_df["LP"] = scores_df["LP"].replace("en-pt-br", "en-pt")
scores_df["LP"] = scores_df["LP"].replace("pt-br-en", "pt-en")
scores_df["LP"] = scores_df["LP"].apply(lambda x: x.upper())
scores_df["Language"] = scores_df["LP"].apply(
    lambda x: x.split("-")[1].upper() if x.startswith("EN") else x.split("-")[0].upper()
)
scores_df["Direction"] = scores_df["LP"].apply(
    lambda x: r"EN$\rightarrow$XX" if x.startswith("EN") else r"XX$\rightarrow$EN"
)

In [ ]:
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

#!wget https://github.com/Phonbopit/sarabun-webfont/raw/master/fonts/thsarabunnew-webfont.ttf
import matplotlib as mpl

#mpl.font_manager.fontManager.addfont("thsarabunnew-webfont.ttf")
#mpl.rc("font", family="TH Sarabun New")

sns.set_style(rc={"patch.force_edgecolor": True, "patch.edgecolor": "black"})
sns.set_style("ticks")

sns.set_style(
    rc={"patch.force_edgecolor": True, "patch.edgecolor": "black", "font": "Palatino"}
)

plt.rcParams["font.family"] = "Palatino"  # change to palatino
plt.rcParams["font.size"] = 10  # change to palatino

cmap = sns.color_palette(["#3953BF", "#b54343"])

for metric in ["COMET", "CHRF"]:
    plt.figure(figsize=(10, 8))

    ax = sns.lineplot(
        data=scores_df,
        x="# of Turns",
        y=metric,
        hue="Language",
        style="Direction",
        linewidth=2.5,
        palette="colorblind",
        markers=["s", "o"],
        # size=10000
        markersize=10,
    )
    ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))

    for spine in ax.spines.values():
        spine.set_visible(False)  # Hide all spines

    # Show only x and y axes
    ax.spines["bottom"].set_visible(True)
    ax.spines["left"].set_visible(True)

    # Increase line width for the visible spines
    ax.spines["bottom"].set_linewidth(2)  # Adjust as needed
    ax.spines["left"].set_linewidth(2)  # Adjust as needed
    # plt.ylim(90, 96)
    if metric == "COMET":
        ax.set_yticks([91, 92, 93, 94, 95])
    elif metric == "CHRF":
        ax.set_yticks([60, 65, 70, 75, 80])
    ax.set_xlim(-0.2, 6.1)
    plt.grid(alpha=0.25)
    #plt.show()

    plt.savefig(f"../plots/turn_scores_{metric}.pdf", dpi=150, bbox_inches="tight")

## contextcomet

In [ ]:
import pandas as pd
import sacrebleu
import numpy as np

split = "test"
dataset = "bcontrast"
lang = "de"
df = pd.read_csv(f"../tacl_results/{dataset}/{split}.en-{lang}.csv", index_col=0)
# mt="comet-best"
mt = "source"

scores = {}
for i in [0, 2, 6, 10, 15]:
    df["outputs"] = read_file(
        f"/mnt/data-poseidon/sweta/chat-translation-generation/wmt24-chat-translation/mbr_outputs/mbr_outputs_test/full_context_empty_sys/TowerInstruct-7B-w-chat-empty-sys/{dataset}_test.en-{lang}/comet_eps_context_{mt}_w{i}.out.txt"
    )
    scores[i] = {}
    df["comet"] = comet_metric.predict(
        [
            {"mt": y, "src": x, "ref": z}
            for x, y, z in zip(
                df["source"].to_list(),
                df["outputs"].to_list(),
                df["reference"].to_list(),
            )
        ],
        batch_size=64,
        gpus=1,
        progress_bar=True,
        devices=[0],
    )["scores"]
    for lp, lp_df in df.groupby("lp"):
        scores[i][lp] = {}
        scores[i][lp]["comet"] = np.mean(lp_df["comet"])
        scores[i][lp]["chrf"] = sacrebleu.corpus_chrf(
            lp_df["outputs"].to_list(), [lp_df["reference"].to_list()]
        ).score

In [ ]:
scores

# DE pronoun analysis

In [ ]:
import sacrebleu
import pandas as pd

split = "test"

model_names = model_names = [
    "baseline",
    "greedy-7b-w-context",
    "greedy-7b-wo-context",
    "greedy-7b-5shot-w-context",
    "greedy-7b-5shot-wo-context",
    "greedy-7b-ft-w-context",
    "greedy-7b-ft-wo-context",
    "mbr",
    "mbr-source",
    "greedy-70b-5shot-w-context",
    "greedy-70b-5shot-wo-context",
]

lang = "de"
df = pd.read_csv(f"paper_results/{split}.en-{lang}.csv")

In [ ]:
pd.set_option("display.max_colwidth", None)

In [ ]:
df[(df.lp == "en-de") & df["tags"].str.contains("pronouns")][
    [
        "tags",
        "source",
        "reference",
        "baseline",
        "greedy-7b-ft-w-context",
        "mbr",
        "mbr-source",
    ]
].head(60)

# Contrastive Decoding 

In [ ]:
import pandas as pd
import sacrebleu

split = "test"
# all_scores = []
for lang in ["de", "fr"]:
    df = pd.read_csv(f"../paper_results/{split}.en-{lang}.csv")
    df["lp"] = df["source_language"] + "-" + df["target_language"]

    fnames = {
        "greedy-7b-ft-contrast-c1_nc0": read_file(
            f"../contrast_decode_tmp/{split}/wmt24_chat_{split}.en-{lang}/c1_nc0.out.txt"
        ),
        "greedy-7b-ft-contrast-c0_nc1": read_file(
            f"../contrast_decode_tmp/{split}/wmt24_chat_{split}.en-{lang}/c0_nc1.out.txt"
        ),
        "greedy-7b-ft-contrast-c1_nc1_b5": read_file(
            f"../contrast_decode_tmp/{split}/wmt24_chat_{split}.en-{lang}/c1_nc1_b5.out.txt"
        ),
        "greedy-7b-ft-contrast-c1_nc1": read_file(
            f"../contrast_decode_tmp/{split}/wmt24_chat_{split}.en-{lang}/c1_nc1.out.txt"
        ),
        "greedy-7b-ft-contrast-c5_nc1": read_file(
            f"../contrast_decode_tmp/{split}/wmt24_chat_{split}.en-{lang}/c5_nc1.out.txt"
        ),
    }

    for mname, file_path in fnames.items():
        df[mname] = fnames[mname]
        df[f"{mname}-comet"] = comet_metric.predict(
            [
                {"mt": y, "src": x, "ref": z}
                for x, y, z in zip(
                    df["source"].to_list(), fnames[mname], df["reference"].to_list()
                )
            ],
            batch_size=64,
            gpus=1,
            progress_bar=True,
            devices=[0],
        )["scores"]

        for lp, lp_df in df.groupby("lp"):
            all_scores.append(
                [
                    lp,
                    mname,
                    sacrebleu.corpus_chrf(
                        lp_df[mname].to_list(), [lp_df["reference"].to_list()]
                    ).score,
                    lp_df[f"{mname}-comet"].mean(),
                ]
            )

In [ ]:
scores_df = pd.DataFrame(all_scores)
scores_df.columns = ["lp", "model", "chrf", "comet"]
scores_df["comet"] = scores_df["comet"] * 100

In [ ]:
for gr, gr_df in scores_df.groupby("model"):
    enxx = (" & ").join(
        [
            f"{gr_df[gr_df.lp=='en-'+lang]['comet'].values[0]:.2f}"
            for lang in ["de", "fr"]
        ]
    )
    xxen = (" & ").join(
        [
            f"{gr_df[gr_df.lp==lang + '-en']['comet'].values[0]:.2f}"
            for lang in ["de", "fr"]
        ]
    )
    print(f"{gr} & {enxx} && {xxen} ")

In [ ]:
for gr, gr_df in scores_df.groupby("model"):
    enxx = (" & ").join(
        [
            f"{gr_df[gr_df.lp=='en-'+lang]['chrf'].values[0]:.2f}"
            for lang in ["de", "fr"]
        ]
    )
    xxen = (" & ").join(
        [
            f"{gr_df[gr_df.lp==lang + '-en']['chrf'].values[0]:.2f}"
            for lang in ["de", "fr"]
        ]
    )
    print(f"{gr} & {enxx} && {xxen} ")

# Sentence-level selection

In [ ]:
def read_file(fname):
    output = []
    with open(fname) as f:
        for line in f:
            output.append(line.strip())
    return output


def get_logprobs(
    df,
    lp,
    context_type,
    model_name="TowerInstruct-7B-w-chat-empty-sys",
    split="dev",
    prob_type="hyp",
    folder="pcxmi",
):
    source_lang = lp.split("-")[0]
    target_lang = lp.split("-")[1].replace("pt", "pt-br")

    if folder == "pcxmi":
        xx_yy_logprobs = list(
            map(
                float,
                read_file(
                    f"../{folder}/{context_type}/mt/wmt24_chat_{split.replace('test', 'test_blind')}.{source_lang}-{target_lang}/{model_name}/mean_log_probs_{prob_type}.txt"
                ),
            )
        )
        yy_xx_logprobs = list(
            map(
                float,
                read_file(
                    f"../{folder}/{context_type}/mt/wmt24_chat_{split.replace('test', 'test_blind')}.{target_lang}-{source_lang}/{model_name}/mean_log_probs_{prob_type}.txt"
                ),
            )
        )
    else:
        xx_yy_logprobs = list(
            map(
                float,
                read_file(
                    f"../{folder}/{context_type}/mt/wmt24_chat_{split}.{source_lang}-{target_lang}/{model_name}/mean_log_probs.txt"
                ),
            )
        )
        yy_xx_logprobs = list(
            map(
                float,
                read_file(
                    f"../{folder}/{context_type}/mt/wmt24_chat_{split}.{target_lang}-{source_lang}/{model_name}/mean_log_probs.txt"
                ),
            )
        )

    assert len(xx_yy_logprobs) + len(yy_xx_logprobs) == len(df)
    logprobs = []
    j, k = 0, 0

    for i, row in df.iterrows():
        if row["lp"] == f"{source_lang}-{target_lang}":
            logprobs.append(xx_yy_logprobs[j])
            j += 1
        else:
            # nl-en and it-en are considered here
            logprobs.append(yy_xx_logprobs[k])
            k += 1
    return logprobs

In [ ]:
import pandas as pd
import sacrebleu

split = "test"
all_scores = []
for lang in [
    "de",
    "fr",
    "ko",
    "nl",
    "pt",
]:
    df = pd.read_csv(f"../paper_results/{split}.en-{lang}.csv")

    df["greedy-7b-ft-w-context-logprob"] = get_logprobs(
        df, f"en-{lang}", "full_context_empty_sys", split=split, prob_type="ref"
    )
    df["greedy-7b-ft-wo-context-logprob"] = get_logprobs(
        df, f"en-{lang}", "no_context_empty_sys", split=split, prob_type="ref"
    )
    df["pcxmi"] = (
        df["greedy-7b-ft-w-context-logprob"] - df["greedy-7b-ft-wo-context-logprob"]
    )

    for col in [
        "no_context_input_on_no_context_output",
        "no_context_input_on_full_context_output",
        "full_context_input_on_no_context_output",
        "full_context_input_on_full_context_output",
    ]:
        df[f"{col}-logprob"] = get_logprobs(
            df,
            f"en-{lang}",
            col,
            model_name="TowerInstruct-v0.2-w-chat-mt-data",
            split=split,
            folder="pcxmi_hyps",
        )

    df["log ld"] = (
        df["full_context_input_on_full_context_output-logprob"]
        - df["no_context_input_on_no_context_output-logprob"]
    )
    df["pcxmi (hyp)"] = (
        df["full_context_input_on_full_context_output-logprob"]
        - df["no_context_input_on_full_context_output-logprob"]
    )

    for measure in ["log ld", "pcxmi", "pcxmi (hyp)"]:
        df[f"greedy-7b-ft-{measure}-select-comet"] = df.apply(
            lambda x: (
                x["greedy-7b-ft-w-context-comet"]
                if x[measure] >= 0
                else x["greedy-7b-ft-wo-context-comet"]
            ),
            axis=1,
        )
        df[f"greedy-7b-ft-{measure}-select"] = df.apply(
            lambda x: (
                x["greedy-7b-ft-w-context"]
                if x[measure] >= 0
                else x["greedy-7b-ft-wo-context"]
            ),
            axis=1,
        )

    df["oracle-select"] = df.apply(
        lambda x: (
            x["greedy-7b-ft-w-context"]
            if x["greedy-7b-ft-w-context-comet"] >= x["greedy-7b-ft-wo-context-comet"]
            else x["greedy-7b-ft-wo-context"]
        ),
        axis=1,
    )
    df["oracle-select-comet"] = df.apply(
        lambda x: (
            x["greedy-7b-ft-w-context-comet"]
            if x["greedy-7b-ft-w-context-comet"] >= x["greedy-7b-ft-wo-context-comet"]
            else x["greedy-7b-ft-wo-context-comet"]
        ),
        axis=1,
    )

    for lp, lp_df in df.groupby("lp"):
        for col in [
            "oracle-select",
            "greedy-7b-ft-log ld-select",
            "greedy-7b-ft-pcxmi (hyp)-select",
            "greedy-7b-ft-pcxmi-select",
            "greedy-7b-ft-w-context",
            "greedy-7b-ft-wo-context",
        ]:
            all_scores.append(
                [
                    lp,
                    col,
                    sacrebleu.corpus_chrf(
                        lp_df[col].to_list(), [lp_df["reference"].to_list()]
                    ).score,
                    lp_df[f"{col}-comet"].mean() * 100,
                ]
            )

In [ ]:
scores_df = pd.DataFrame(all_scores)
scores_df.columns = ["lp", "model", "chrf", "comet"]

In [ ]:
for gr, gr_df in scores_df.groupby("model"):
    gr_df["type"] = gr_df["lp"].apply(
        lambda x: "en-xx" if x.split("-")[0] == "en" else "xx-en"
    )
    print(
        f"{gr} & {gr_df[gr_df.type=='en-xx']['chrf'].mean():.2f} & {gr_df[gr_df.type=='en-xx']['comet'].mean():.2f} && "
        f"{gr_df[gr_df.type=='xx-en']['chrf'].mean():.2f} & {gr_df[gr_df.type=='xx-en']['comet'].mean():.2f}"
    )

# Number of turns - CXMI

In [ ]:
def read_file(fname):
    output = []
    with open(fname) as f:
        for line in f:
            output.append(line.strip())
    return output


def get_logprobs_turns(
    df,
    lp,
    context_type,
    model_name="TowerInstruct-7B-w-chat-empty-sys",
    split="dev",
    prob_type="hyp",
    folder="pcxmi",
):
    source_lang = lp.split("-")[0]
    target_lang = lp.split("-")[1].replace("pt", "pt-br")

    xx_yy_logprobs = list(
        map(
            float,
            read_file(
                f"../{folder}/{context_type}/mt/wmt24_chat_{split}.{source_lang}-{target_lang}/{model_name}/mean_log_probs_{prob_type}.txt"
            ),
        )
    )
    yy_xx_logprobs = list(
        map(
            float,
            read_file(
                f"../{folder}/{context_type}/mt/wmt24_chat_{split}.{target_lang}-{source_lang}/{model_name}/mean_log_probs_{prob_type}.txt"
            ),
        )
    )

    assert len(xx_yy_logprobs) + len(yy_xx_logprobs) == len(df)
    logprobs = []
    j, k = 0, 0

    for i, row in df.iterrows():
        if row["lp"] == f"{source_lang}-{target_lang}":
            logprobs.append(xx_yy_logprobs[j])
            j += 1
        else:
            # nl-en and it-en are considered here
            logprobs.append(yy_xx_logprobs[k])
            k += 1
    return logprobs

In [ ]:
import pandas as pd

split = "test"

for lang in ["de", "pt", "ko", "nl", "fr"]:
    df = pd.read_csv(f"../tacl_results/wmt24_chat/{split}.en-{lang}.csv")
    df["lp"] = df["source_language"] + "-" + df["target_language"]

    fnames = {}
    for turn in [
        "full_context_empty_sys_2_turns",
        "full_context_empty_sys_6_turns",
        "full_context_empty_sys_10_turns",
        "full_context_empty_sys_15_turns",
        "full_context_empty_sys_20_turns",
    ]:

        df[f"{turn}-logprob"] = get_logprobs_turns(
            df,
            f"en-{lang}",
            turn,
            model_name="TowerInstruct-v0.2-w-chat-mt-data",
            split=split,
            prob_type="ref",
        )
        df["no-context-logprob"] = get_logprobs_turns(
            df, f"en-{lang}", "no_context_empty_sys", split=split, prob_type="ref"
        )
        df[f"{turn}-pcxmi"] = df[f"{turn}-logprob"] - df["no-context-logprob"]

    df[f"full_context_empty_sys-logprob"] = get_logprobs_turns(
        df, f"en-{lang}", "full_context_empty_sys", split=split, prob_type="ref"
    )
    df["no-context-logprob"] = get_logprobs_turns(
        df, f"en-{lang}", "no_context_empty_sys", split=split, prob_type="ref"
    )
    df[f"full_context_empty_sys-pcxmi"] = (
        df[f"{turn}-logprob"] - df["no-context-logprob"]
    )

    df.to_csv(
        f"../tacl_results/wmt24_chat/{split}.en-{lang}.pcxmi.turns.csv", index=None
    )

In [ ]:
import sacrebleu
import pandas as pd

split = "test"

model_names = {
    2: "full_context_empty_sys_2_turns",
    6: "full_context_empty_sys_6_turns",
    10: "full_context_empty_sys_10_turns",
    15: "full_context_empty_sys_15_turns",
    20: "full_context_empty_sys_20_turns",
    100: "full_context_empty_sys",
}

all_scores = []
for lang in ["de", "fr", "pt", "ko", "nl"]:
    df = pd.read_csv(f"../tacl_results/wmt24_chat/{split}.en-{lang}.pcxmi.turns.csv")
    df.fillna("", inplace=True)

    for lp, lp_df in df.groupby("lp"):
        for t, mname in model_names.items():
            all_scores.append([lp, t, lp_df[f"{mname}-pcxmi"].mean()])


scores_df = pd.DataFrame(all_scores)
scores_df.columns = ["LP", "#Turns", "PCXMI"]

scores_df["#Turns"] = scores_df["#Turns"].astype(str)
scores_df["#Turns"] = scores_df["#Turns"].replace("100", "Full")
scores_df["#Turns"] = scores_df["#Turns"].replace("0", "No")
scores_df["LP"] = scores_df["LP"].replace("en-pt-br", "en-pt")
scores_df["LP"] = scores_df["LP"].replace("pt-br-en", "pt-en")
scores_df["LP"] = scores_df["LP"].apply(lambda x: x.upper())

In [ ]:
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

# !wget https://github.com/Phonbopit/sarabun-webfont/raw/master/fonts/thsarabunnew-webfont.ttf
import matplotlib as mpl

mpl.font_manager.fontManager.addfont("thsarabunnew-webfont.ttf")
mpl.rc("font", family="TH Sarabun New")

sns.set_style(rc={"patch.force_edgecolor": True, "patch.edgecolor": "black"})
sns.set_style("ticks")

plt.rcParams["font.size"] = 10  # change to palatino
# # Setting the Seaborn context to adjust font sizes
sns.set_context("notebook", font_scale=2.2)

plt.figure(figsize=(10, 8))
metric = "PCXMI"

ax = sns.lineplot(
    data=scores_df, x="#Turns", y=metric, hue="LP", marker="o", linewidth=2.5
)
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))

for spine in ax.spines.values():
    spine.set_visible(False)  # Hide all spines

# Show only x and y axes
ax.spines["bottom"].set_visible(True)
ax.spines["left"].set_visible(True)

# Increase line width for the visible spines
ax.spines["bottom"].set_linewidth(2)  # Adjust as needed
ax.spines["left"].set_linewidth(2)  # Adjust as needed
# plt.ylim(90, 96)
# plt.show()

plt.savefig(f"../plots/turn_scores_{metric}.pdf", dpi=150, bbox_inches="tight")

# Inference Number of turns ablation

In [ ]:
from comet import download_model, load_from_checkpoint

comet_metric = load_from_checkpoint(download_model("Unbabel/wmt22-comet-da"))

In [ ]:
import pandas as pd


def read_file(fname):
    output = []
    with open(fname) as f:
        for line in f:
            output.append(line.strip())
    return output


split = "test"
context_type = "comet-best"

for lang in ["de", "pt", "ko", "nl", "fr"]:
    df = pd.read_csv(f"../tacl_results/wmt24_chat/{split}.en-{lang}.csv")
    df["lp"] = df["source_language"] + "-" + df["target_language"]

    fnames = {}
    for turn in [0, 2, 6, 10, 15]:
        for context_type in ["source", "comet-best"]:
            fnames[f"{turn}-{context_type}"] = read_file(
                f"../mbr_outputs/mbr_outputs_{split}/full_context_empty_sys/TowerInstruct-7B-w-chat-empty-sys/wmt24_chat_{split}.en-{lang}/comet_eps_context_{context_type}_w{turn}.out.txt"
            )

    for mname, file_path in fnames.items():
        df[mname] = fnames[mname]
        df[f"{mname}-comet"] = comet_metric.predict(
            [
                {"mt": y, "src": x, "ref": z}
                for x, y, z in zip(
                    df["source"].to_list(), fnames[mname], df["reference"].to_list()
                )
            ],
            batch_size=256,
            gpus=1,
            progress_bar=True,
            devices=[0],
        )["scores"]
    df.to_csv(f"../tacl_results/wmt24_chat/{split}.en-{lang}.turns.mbr.csv", index=None)

In [ ]:
import sacrebleu
import pandas as pd

split = "test"


all_scores = []
for lang in ["de", "fr", "pt", "ko", "nl"]:
    df = pd.read_csv(f"../tacl_results/wmt24_chat/{split}.en-{lang}.turns.mbr.csv")
    df.fillna("", inplace=True)

    for lp, lp_df in df.groupby("lp"):
        for turn in [0, 2, 6, 10, 15]:
            for context_type in ["source", "comet-best"]:
                all_scores.append(
                    [
                        lp,
                        f"{turn}",
                        f"{context_type}",
                        sacrebleu.corpus_chrf(
                            lp_df[f"{turn}-{context_type}"].to_list(),
                            [lp_df["reference"].to_list()],
                        ).score,
                        lp_df[f"{turn}-{context_type}-comet"].mean(),
                    ]
                )

scores_df = pd.DataFrame(all_scores)
scores_df.columns = ["LP", "#Turns", "Context", "ChrF", "Comet"]

scores_df["Comet"] = scores_df["Comet"] * 100

scores_df["#Turns"] = scores_df["#Turns"].astype(str)
scores_df["#Turns"] = scores_df["#Turns"].replace("100", "Full")
scores_df["#Turns"] = scores_df["#Turns"].replace("0", "No")
scores_df["LP"] = scores_df["LP"].replace("en-pt-br", "en-pt")
scores_df["LP"] = scores_df["LP"].replace("pt-br-en", "pt-en")
scores_df["LP"] = scores_df["LP"].apply(lambda x: x.upper())

In [ ]:
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

# !wget https://github.com/Phonbopit/sarabun-webfont/raw/master/fonts/thsarabunnew-webfont.ttf
import matplotlib as mpl

mpl.font_manager.fontManager.addfont("thsarabunnew-webfont.ttf")
mpl.rc("font", family="TH Sarabun New")

sns.set_style(rc={"patch.force_edgecolor": True, "patch.edgecolor": "black"})
sns.set_style("ticks")

plt.rcParams["font.size"] = 10  # change to palatino
# # Setting the Seaborn context to adjust font sizes
sns.set_context("notebook", font_scale=2.2)

for metric in ["Comet", "ChrF"]:
    plt.figure(figsize=(10, 8))

    ax = sns.lineplot(
        data=scores_df,
        x="#Turns",
        y=metric,
        hue="LP",
        marker="o",
        linewidth=2.5,
        style="Context",
    )
    ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))

    for spine in ax.spines.values():
        spine.set_visible(False)  # Hide all spines

    # Show only x and y axes
    ax.spines["bottom"].set_visible(True)
    ax.spines["left"].set_visible(True)

    # Increase line width for the visible spines
    ax.spines["bottom"].set_linewidth(2)  # Adjust as needed
    ax.spines["left"].set_linewidth(2)  # Adjust as needed
    # plt.ylim(90, 96)
    plt.show()

    # plt.savefig(f'../plots/turn_scores_mbr_{metric}.pdf', dpi=150, bbox_inches='tight')